# NLP Assignment 1 — Fine-grained emotion detection with GoEmotions
**Phase 1, part 2 of 2: the cleaning pipeline (Tasks 4–5)**

Team: *add names / IDs here* · Course: *add course / section* · Date: *add date*

This notebook builds on the findings of part 1 (`01_exploration_and_tokenization.ipynb`: statistics, messy-text inventory, tokenizers and regex patterns). Every section follows the same pattern: *measure → decide → verify*.

| Section | Content |
|---|---|
| Task 4 | the cleaning pipeline: design, step-by-step trace, case folding, stop words, stemming vs lemmatization, before/after |
| Task 5 | one command that reproduces the cleaned dataset, verified against this notebook |

The reusable code lives in `src/` (`load_data.py`, `cleaning.py`, `run_cleaning.py`); this notebook only *calls* it, so what you see here is exactly what the script runs. The first code cell below is the same set-up cell as in part 1 (imports, project root, data loading), so this notebook runs on its own.

In [ ]:
import re, sys, time, json, subprocess, unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from IPython.display import display

def find_project_root():
    """Locate the folder that contains src/cleaning.py, wherever Jupyter was started:
    the repo root, notebooks/, a parent folder of the repo, or a sub-folder of it."""
    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        for candidate in (base, base / "goemotions-nlp"):
            if (candidate / "src" / "cleaning.py").exists():
                return candidate
    raise FileNotFoundError(
        f"Could not find the project's src/ folder from {start}.\n"
        "This notebook needs the whole project, not just the .ipynb file: clone the repository "
        "(or unzip goemotions-nlp.zip) and open notebooks/02_cleaning_pipeline.ipynb from inside it "
        "(on Colab/Kaggle, clone or upload the project, then %cd goemotions-nlp/notebooks first)."
    )

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

from src import cleaning as C
from src.load_data import load_goemotions, emotion_names, multi_hot

for resource in ("punkt_tab", "stopwords", "wordnet"):
    nltk.download(resource, quiet=True)

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 170)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 42

df = load_goemotions()            
NAMES = emotion_names()
Y = multi_hot(df, NAMES)          
print(f"{len(df):,} comments | {len(NAMES)} labels | splits: {df.split.value_counts().to_dict()}")

SHUFFLED = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

def pick(pattern, n=1, max_len=130, flags=0):
    """First n real comments (in the fixed shuffle) matching a regex."""
    rx = re.compile(pattern, flags)
    hits = SHUFFLED[SHUFFLED.text.map(lambda t: bool(rx.search(t)) and len(t) <= max_len)]
    return hits.text.head(n).tolist()

CAPS = r"(?<![\[A-Za-z])[A-Z]{3,}(?![A-Za-z\]])"

df.head()

54,263 comments | 28 labels | splits: {'train': 43410, 'test': 5427, 'validation': 5426}


,comment_id,split,text,label_ids,label_names
0,eebbqej,train,My favourite food is anything I didn't have to cook myself.,27,neutral
1,ed00q6i,train,"Now if he does off himself, everyone will think hes having a laugh screwing with people instead of actuall...",27,neutral
2,eezlygj,train,WHY THE FUCK IS BAYLESS ISOING,2,anger
3,ed7ypvh,train,To make her feel threatened,14,fear
4,ed0bdzj,train,Dirty Southern Wankers,3,annoyance


---
# Task 4 — The cleaning pipeline

## 4.1 Design: what runs, in what order, and why

The pipeline produces **two outputs from one code path** (`src/cleaning.py`), because the two kinds of model we will compare in Phase 2 want opposite things:

* **`text_norm`** — normalised but still natural text, for **BERT**. A pretrained model has seen natural sentences; stripping stop words or lemmatizing would put it *out of distribution* and throw away information it knows how to use.
* **`text_clean`** — additionally tokenized, lemmatized and stop-word filtered, for **TF-IDF / bag-of-words models**, which gain from a smaller, less sparse vocabulary.

| Step | What it does | Why *here* in the order |
|---|---|---|
| 1. Unicode (NFKC; strip combining / zero-width chars) | folds full-width `！？` and `…`, removes Zalgo and invisible characters | first, because every later regex assumes canonical characters |
| 2. HTML entities, curly quotes, dashes | `’` → `'`, `&amp;` → `&` | before tokenization so `don’t` and `don't` are one word |
| 3. Mask entities, map emoticons | `[NAME]` → `<name>`, `r/x` → `<subreddit>`, digits → `<num>`, `:)` → `<smile>` | **before** lower-casing (placeholders are upper-case) and before punctuation is touched (emoticons *are* punctuation) |
| 4. Case folding | lower-case | after masking; signals were measured on the raw text (4.3) |
| 5. Squeeze + repair + isolate | `sooooo` → `soo`, `!!!` → `!`, `...don't` → `... don't`, emoji separated | after case folding so the elongation regex needs only `a–z` |
| 6. Whitespace | collapse and strip | last of the string-level steps → **`text_norm`** |
| 7. Tokenize + lemmatize (spaCy) | `didn't` → `do` `not`, `annoyed` stays, `was` → `be` | needs the normalised string; lemmas, not stems (see 4.5) |
| 8. Custom stop-word filter | drops function words but **keeps** negations, question words, I/you pronouns, degree words | after lemmatization so `n't` has already become `not` (4.4) |
| 9. Keep only informative tokens | alphabetic lemmas, `<…>` placeholders, emoji, and `! ? ...`; if *everything* would be removed, fall back so no comment becomes empty → **`text_clean`** | last |

## 4.2 Watch the pipeline work: three real comments, step by step
A `*changed*` flag shows which steps actually did something to each comment.

In [2]:
def trace(text):
    raw, current = text, text
    print(f"RAW  {raw!r}")
    for name, fn in C.NORMALIZATION_STEPS:
        new = fn(current)
        print(f"\n{name}  [{'changed' if new != current else 'no change'}]\n   {new!r}")
        current = new
    print(f"\n7. tokenise + lemmatise + custom stop words  ->  {C.clean_text(raw)!r}\n" + "-" * 90)

for text in (pick(r"\[NAME\].*\.\.\..*\u2019", max_len=150) or pick(r"\[NAME\].*\.\.\.", max_len=150))[:1] \
          + pick(CAPS + r".*[!?]", max_len=90)[:1] \
          + pick(r"[a-z]:\)", max_len=90)[:1]:
    trace(text)

RAW  '[NAME] on a pretzel. That is just...fucked up. Those comments are absolutely...I can’t think of anything awful enough to describe them.'

1. unicode (NFKC, strip invisible/combining chars)  [no change]
   '[NAME] on a pretzel. That is just...fucked up. Those comments are absolutely...I can’t think of anything awful enough to describe them.'

2. HTML entities + curly quotes/dashes  [changed]
   "[NAME] on a pretzel. That is just...fucked up. Those comments are absolutely...I can't think of anything awful enough to describe them."

3. mask names/religion/urls/subreddits/numbers, map emoticons  [changed]
   " <name>  on a pretzel. That is just...fucked up. Those comments are absolutely...I can't think of anything awful enough to describe them."

4. case folding  [changed]
   " <name>  on a pretzel. that is just...fucked up. those comments are absolutely...i can't think of anything awful enough to describe them."

5. squeeze elongation/punctuation, fix glued punctuation, isolate emoj


7. tokenise + lemmatise + custom stop words  ->  '<name> pretzel just ... fuck comment absolutely ... i not think anything awful enough describe'
------------------------------------------------------------------------------------------
RAW  'I second that motion...with A VENGEANCE!!'

1. unicode (NFKC, strip invisible/combining chars)  [no change]
   'I second that motion...with A VENGEANCE!!'

2. HTML entities + curly quotes/dashes  [no change]
   'I second that motion...with A VENGEANCE!!'

3. mask names/religion/urls/subreddits/numbers, map emoticons  [no change]
   'I second that motion...with A VENGEANCE!!'

4. case folding  [changed]
   'i second that motion...with a vengeance!!'

5. squeeze elongation/punctuation, fix glued punctuation, isolate emoji  [changed]
   'i second that motion ... with a vengeance!'

6. whitespace  [no change]
   'i second that motion ... with a vengeance!'

7. tokenise + lemmatise + custom stop words  ->  'i second motion ... vengeance !'
-----------

**Reading the traces.**
Comment 1: the curly apostrophe is straightened in step 2, `[NAME]` is masked in step 3, and the glued ellipses (`just...fucked`) are separated in step 5 — without that the tokenizer would keep `...fucked` as one useless token. `can't` ends as the single token **`not`**.
Comment 2: the case is folded in step 4, so `A VENGEANCE` loses its capitals — which is why shouting is measured *beforehand* (next section); `!!` becomes `!`, which survives as a token.
Comment 3: `close:)` has the smiley glued to a word; steps 2–3 turn it into `<smile>`, so the emotion cue survives instead of being deleted as "punctuation".

## 4.3 Case folding: is capitalisation (and `!` / `?`) a signal?
Before lower-casing destroys it, we measure whether it carries emotion information.

In [3]:
sig = pd.DataFrame([C.extract_signals(t) for t in df.text], index=df.index)
Yb = Y.astype(bool)
rate = lambda s, mask=slice(None): round(100 * (s[mask] > 0).mean(), 1)
table = pd.DataFrame({
    "has ALL-CAPS word %": {l: rate(sig.caps_words, Yb[l]) for l in NAMES},
    "has '!' %":           {l: rate(sig.exclaims,   Yb[l]) for l in NAMES},
    "has '?' %":           {l: rate(sig.questions,  Yb[l]) for l in NAMES},
})
display(pd.DataFrame({"has ALL-CAPS word %": [rate(sig.caps_words)], "has '!' %": [rate(sig.exclaims)],
                      "has '?' %": [rate(sig.questions)]}, index=["all comments"]))
for col in table:
    print(f"{col:22s} highest:", table[col].sort_values(ascending=False).head(4).to_dict())

,has ALL-CAPS word %,has '!' %,has '?' %
all comments,9.2,13.9,10.5


has ALL-CAPS word %    highest: {'grief': 16.7, 'anger': 13.7, 'excitement': 11.3, 'amusement': 11.2}
has '!' %              highest: {'excitement': 42.7, 'gratitude': 37.9, 'pride': 28.9, 'joy': 26.9}
has '?' %              highest: {'curiosity': 80.7, 'confusion': 50.3, 'surprise': 15.6, 'anger': 10.9}


**What this tells us.** Surface style is correlated with label, sometimes strongly:

* `?` appears in 10.5% of all comments but in **80.7% of `curiosity`** and **50.3% of `confusion`** — a question mark is close to a defining feature of those two classes;
* `!` appears in 13.9% of all comments but in **42.7% of `excitement`** and **37.9% of `gratitude`**;
* ALL-CAPS words appear in 9.2% of all comments but in **13.7% of `anger`** (and 16.7% of `grief`, though `grief` has only 96 comments, so treat that figure cautiously).

**Decision.** Lower-case the text (it removes the case variants — "The"/"the"/"THE" — and cuts the vocabulary from 65,523 to 56,964 types, −13.1%), **but** (a) keep `!`, `?` and `...` as tokens instead of stripping punctuation, and (b) store `caps_words`, `exclaims` and `questions` as separate numeric columns measured on the raw text, so capitalisation is not lost for Phase 2. The capital-letter effect is modest next to the `?`/`!` effect, which is why the numeric columns are an optional extra rather than a main feature.

## 4.4 Stop words: the standard list is harmful on short emotional text

In [ ]:
default, custom = C.default_stopwords(), C.custom_stopwords()
print(f"NLTK default list: {len(default)} words | our custom list: {len(custom)} words")
print("deliberately KEPT (in NLTK's list, not in ours):", sorted(default & C.KEEP_WORDS))

norm_raw = df.text.str.lower().str.replace("\u2019", "'")
tokset = norm_raw.map(lambda s: set(re.findall(r"[a-z]+(?:'[a-z]+)?", s)))
Yb = Y.astype(bool)

def by_label(has):
    has = np.asarray(has, dtype=bool)
    per_label = {l: round(float(100 * has[Yb[l].to_numpy()].mean()), 1) for l in NAMES}
    return round(float(100 * has.mean()), 1), dict(sorted(per_label.items(), key=lambda kv: -kv[1])[:4])

DELETED_NEG = (C.NEGATIONS & default) | {w for w in default if w.endswith("n't")}
groups = {
    "negations the default list deletes": tokset.map(lambda s: bool(s & DELETED_NEG)),
    "question words":                     tokset.map(lambda s: bool(s & C.QUESTION_WORDS)),
    "I / you pronouns":                   tokset.map(lambda s: bool(s & C.PERSONAL_PRONOUNS)),
}
for group, has in groups.items():
    overall, top = by_label(has)
    print(f"\n{group}: in {overall}% of ALL comments; highest labels -> {top}")

print()
for s in ["why would you not tell me", "i don't love you", "this is not what i expected", "how could you"]:
    toks = s.split()
    print(f"{s!r:32s} default -> {[w for w in toks if w not in default]!s:30s} custom -> {[w for w in toks if w not in custom]}")

NLTK default list: 198 words | our custom list: 167 words
deliberately KEPT (in NLTK's list, not in ours): ['how', 'i', 'just', 'me', 'more', 'most', 'my', 'myself', 'no', 'nor', 'not', 'only', 'our', 'ourselves', 'should', 'so', 'such', 'too', 'very', 'we', 'what', 'when', 'where', 'which', 'who', 'whom', 'why', 'you', 'your', 'yours', 'yourself']



negations the default list deletes: in 22.4% of ALL comments; highest labels -> {'disapproval': 62.7, 'confusion': 43.5, 'relief': 35.7, 'realization': 29.8}

question words: in 17.6% of ALL comments; highest labels -> {'curiosity': 46.1, 'confusion': 43.6, 'surprise': 24.1, 'annoyance': 21.9}

I / you pronouns: in 55.4% of ALL comments; highest labels -> {'desire': 83.5, 'love': 82.2, 'remorse': 74.4, 'pride': 73.2}

'why would you not tell me'      default -> ['would', 'tell']              custom -> ['why', 'would', 'you', 'not', 'tell', 'me']
"i don't love you"               default -> ['love']                       custom -> ['i', 'love', 'you']
'this is not what i expected'    default -> ['expected']                   custom -> ['not', 'what', 'i', 'expected']
'how could you'                  default -> ['could']                      custom -> ['how', 'could', 'you']


**What this tells us.** The stock NLTK list (198 words) was designed for long documents. Here:

* it deletes negation words in **22.4% of all comments** — and in **62.7% of `disapproval`** comments, 43.5% of `confusion`, 35.7% of `relief`;
* it deletes question words (`why`, `how`, `what`…), which appear in 46.1% of `curiosity` and 43.6% of `confusion` comments versus 17.6% overall;
* it deletes first/second-person pronouns, which appear in 55.4% of all comments and in 83.5% of `desire` and 82.2% of `love` ("I want you", "I love you");
* worst of all, it can *invert* a sentence: `"i don't love you"` becomes `love`, and `"why would you not tell me"` becomes `would tell`.

**Decision.** Use a *custom* list: NLTK's list **minus 31 words** that carry emotion (negations, question words, `i`/`you` pronouns, degree words such as `very`, `too`, `so`) — 167 words are still removed. The cost is a somewhat larger vocabulary and some high-frequency features (`i`, `you`), but TF-IDF down-weights frequent words automatically, whereas deleted information cannot be recovered.

## 4.5 Stemming vs lemmatization
We apply **one** of them to the corpus; this section shows both and explains the choice.

In [5]:
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.corpus import wordnet

porter, snowball, nlp = PorterStemmer(), SnowballStemmer("english"), C.get_nlp()
demo = nlp("She was annoyed and disgusted; their arguments were annoying, but he was caring, careless and happier than ever.")
display(pd.DataFrame([{"token": t.text, "Porter": porter.stem(t.lower_), "Snowball": snowball.stem(t.lower_),
                       "spaCy lemma": t.lemma_.lower()} for t in demo if t.is_alpha]).set_index("token").T)

sample = SHUFFLED.text.head(10_000).map(C.normalize_text).tolist()
surface, lemmas, stems, groups = set(), set(), set(), {}
for doc in nlp.pipe(sample, batch_size=512):
    for t in doc:
        if t.is_alpha:
            surface.add(t.lower_); lemmas.add(t.lemma_.lower())
            st = porter.stem(t.lower_); stems.add(st); groups.setdefault(st, set()).add(t.lower_)

in_wordnet = lambda vocab: round(100 * np.mean([bool(wordnet.synsets(w)) for w in vocab]), 1)
display(pd.DataFrame({"distinct forms": [len(surface), len(lemmas), len(stems)],
                      "% found in WordNet": [in_wordnet(surface), in_wordnet(lemmas), in_wordnet(stems)]},
                     index=["surface forms (lower-cased)", "spaCy lemmas", "Porter stems"]))
print("Porter stems that merge the most different surface words:")
for st, forms in sorted(groups.items(), key=lambda kv: -len(kv[1]))[:8]:
    print(f"  {st!r:12} <- {sorted(forms)[:12]}")

token,She,was,annoyed,and,disgusted,their,arguments,were,annoying,but,he,was,caring,careless,and,happier,than,ever
Porter,she,wa,annoy,and,disgust,their,argument,were,annoy,but,he,wa,care,careless,and,happier,than,ever
Snowball,she,was,annoy,and,disgust,their,argument,were,annoy,but,he,was,care,careless,and,happier,than,ever
spaCy lemma,she,be,annoyed,and,disgusted,their,argument,be,annoying,but,he,be,care,careless,and,happy,than,ever


,distinct forms,% found in WordNet
surface forms (lower-cased),11812,82.8
spaCy lemmas,9469,78.9
Porter stems,8515,53.0


Porter stems that merge the most different surface words:
  'gener'      <- ['general', 'generality', 'generalization', 'generalizations', 'generally', 'generate', 'generation', 'generations', 'generic', 'generous']
  'anim'       <- ['animal', 'animals', 'animated', 'animation', 'animations', 'animator', 'anime']
  'protect'    <- ['protect', 'protected', 'protecting', 'protection', 'protections', 'protectiveness', 'protects']
  'support'    <- ['support', 'supported', 'supporter', 'supporters', 'supporting', 'supportive', 'supports']
  'commun'     <- ['communicate', 'communicated', 'communication', 'communications', 'communism', 'communities', 'community']
  'love'       <- ['love', 'loved', 'lovely', 'loves', 'loving', 'lovings']
  'expect'     <- ['expect', 'expectation', 'expectations', 'expected', 'expecting', 'expects']
  'hope'       <- ['hope', 'hoped', 'hopeful', 'hopefully', 'hopes', 'hoping']


**What this tells us.**

* Stemmers cut suffixes by rule, so the output is often not a word: Porter turns `was` into `wa`, and **only 53% of Porter stems are real English words** (WordNet), versus 79% of spaCy lemmas and 83% of the original surface forms.
* Stemmers over-merge: `gener` ← *general, generous, generate, generic*; `commun` ← *communism, community, communication*; `anim` ← *animal, anime, animated*. Words with different meanings — sometimes different *sentiment* — end up as one feature.
* Lemmatization uses part-of-speech information and a dictionary: `was` → `be`, `happier` → `happy`, `arguments` → `argument`, and `n't` → `not`. The result stays readable and inspectable.

**An honest trade-off.** Stemming is more aggressive: 8,515 distinct stems vs 9,469 lemmas from 11,812 surface forms (10,000-comment sample). Notice that spaCy keeps `annoyed` and `annoying` *apart* (both adjectives) whereas Porter merges them into `annoy` — for the `annoyance` class that merge would probably *help* a sparse TF-IDF model. We still choose **lemmatization** because it produces real words, resolves irregular forms and contractions correctly and merges less aggressively, which is the safer default for 13-word texts; its costs are speed (~45 s for the whole corpus, acceptable) and dependence on the POS tagger's accuracy on noisy text. Because stemming is not clearly worse for sparse models, **Phase 2 can run it as an ablation** (swap one line in `src/cleaning.py`).

## 4.6 Apply the pipeline to the whole corpus and verify it

In [6]:
t0 = time.time()
cleaned = C.clean_corpus(df.text)
print(f"cleaned {len(cleaned):,} comments in {time.time() - t0:.0f}s")
out = pd.concat([df, cleaned], axis=1)

stages = {"raw text (cased)": out.text, "raw text, lower-cased": out.text.str.lower(),
          "text_norm": out.text_norm, "text_clean": out.text_clean}
rows = []
for name, series in stages.items():
    v = C.vocabulary(series)
    rows.append({"stage": name, "vocabulary (types)": len(v), "tokens": sum(v.values()),
                 "mean words / comment": round(series.str.split().str.len().mean(), 2)})
vt = pd.DataFrame(rows).set_index("stage")
vt["types vs raw %"] = (100 * (vt["vocabulary (types)"] / vt.loc["raw text (cased)", "vocabulary (types)"] - 1)).round(1)
display(vt)

raw_v, clean_v = C.vocabulary(out.text), C.vocabulary(out.text_clean)
display(pd.DataFrame({"raw": [w for w, _ in raw_v.most_common(15)],
                      "text_clean": [w for w, _ in clean_v.most_common(15)]}).T)

for r in out.sample(6, random_state=7).itertuples():
    print(f"raw  : {r.text!r}\nnorm : {r.text_norm!r}\nclean: {r.text_clean!r}\n")

cleaned 54,263 comments in 35s


,vocabulary (types),tokens,mean words / comment,types vs raw %
stage,,,,
raw text (cased),65523,695858,12.82,0.0
"raw text, lower-cased",56964,695858,12.82,-13.1
text_norm,50699,707084,13.03,-22.6
text_clean,22405,475269,8.76,-65.8


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
raw,the,I,to,a,and,is,you,of,that,[NAME],in,for,it,this,was
text_clean,i,you,not,<name>,!,?,<num>,my,get,so,just,like,...,what,your


raw  : 'He’s a coward, but no need to ruin his life. You should feel sorry for him. '
norm : "he's a coward, but no need to ruin his life. you should feel sorry for him."
clean: 'coward no need ruin life you should feel sorry'

raw  : "It's so freaky. I'm a 40 something and the stock photos and really odd taglines are just baffling to me."
norm : "it's so freaky. i'm a <num> something and the stock photos and really odd taglines are just baffling to me."
clean: 'so freaky i <num> something stock photo really odd tagline just baffle i'

raw  : 'It sucks it didn’t work out good luck in cincy and thanks for that Alcs game'
norm : "it sucks it didn't work out good luck in cincy and thanks for that alcs game"
clean: 'suck not work good luck cincy thank alcs game'

raw  : 'Favourite month??!!'
norm : 'favourite month?!'
clean: 'favourite month ? !'

raw  : 'Article says she was present when this happened sometimes and that she was physically and psychologically abusive as well'
norm : 'artic

In [ ]:
print("empty text_clean:", int((out.text_clean.str.len() == 0).sum()), "comments")

NEG_RX = re.compile(r"\b(?:no|not|never|nor|none|nobody|nothing|neither|nowhere|cannot)\b|n't\b")
has_neg_raw = norm_raw.map(lambda t: bool(NEG_RX.search(t)))
has_neg_clean = out.text_clean.map(lambda t: bool(set(t.split()) & C.NEGATIONS))
print(f"comments containing a negation   raw text: {100 * has_neg_raw.mean():.1f}%   text_clean: {100 * has_neg_clean.mean():.1f}%")
print(f"raw negations missing from text_clean: {int((has_neg_raw & ~has_neg_clean).sum())} comments")

check = out.sample(300, random_state=1)
assert all(C.clean_text(t) == c for t, c in zip(check.text, check.text_clean)), "single vs batch differ"
assert C.clean_corpus(check.text).equals(out.loc[check.index, ["text_norm", "text_clean"]]), "not deterministic"
assert len(out) == len(df) and (out.label_ids == df.label_ids).all() and (out.split == df.split).all()
print("checks passed: single == batch, deterministic, no rows dropped, labels and splits untouched")

empty text_clean: 0 comments
comments containing a negation   raw text: 26.6%   text_clean: 27.8%
raw negations missing from text_clean: 6 comments


checks passed: single == batch, deterministic, no rows dropped, labels and splits untouched


**Before vs after.**

| stage | vocabulary (types) | change vs raw |
|---|---|---|
| raw, cased | 65,523 | — |
| raw, lower-cased | 56,964 | −13.1% |
| `text_norm` | 50,699 | −22.6% |
| `text_clean` | 22,405 | **−65.8%** |

The tokens go from 695,858 to 475,269 and the average comment shrinks from 12.8 to 8.8 words; most of the drop happens between `text_norm` and `text_clean` (lemmatization, stop words, dropping non-alphabetic tokens); we did not isolate how much each contributes. No comment became empty, and no row, label or split assignment changed.

**The checks that mattered.**

* *Negation survives.* 26.6% of raw comments contain a negation and 27.8% of `text_clean` rows do. It is slightly *higher* after cleaning because spaCy recovers informal spellings such as `dont` / `isnt` as `not`. When we first ran this check it exposed a real bug: 16 comments lost a negation because it was glued to punctuation (`vomitting,can't`, `...don't`); fixing `step_squeeze` brought that down to 6 comments: five malformed words (`isen't`, `Istn't`, `Coloradon't`, `>on't`, `don’t-ee`) and one case of Reddit's superscript markup (`^^^not`) — a known remaining edge case we did not special-case.
* *One code path.* `clean_text(single)` equals `clean_corpus(batch)` and the output is deterministic.

**What is lost (stated openly).** Capitalisation (kept only as the numeric `caps_words`), the actual digits and names (replaced by `<num>`, `<name>`), and the exact spelling of emphasis. We judged these to carry little label information relative to the vocabulary they bloat; Phase 2 can test that assumption. Two side effects of the spaCy lemmatizer are also worth knowing: `can't`, `won't` and `don't` all end up as the single token `not` (the modal `can` / `will` is dropped as a stop word), and `best` becomes `good`, so degree is lost; ordinals such as `1st` become `<num> st`.

---
# Task 5 — One command to rebuild the cleaned dataset

In [8]:
proc = subprocess.run([sys.executable, "-m", "src.run_cleaning"], cwd=ROOT, capture_output=True, text=True)
print(proc.stdout[-1500:])
assert proc.returncode == 0, proc.stderr

saved = pd.read_csv(ROOT / "data" / "processed" / "goemotions_clean.csv", keep_default_na=False,
                    dtype={"comment_id": str, "label_ids": str})
print(saved.shape, saved.columns.tolist())
assert len(saved) == len(df)
assert saved.split.value_counts().to_dict() == df.split.value_counts().to_dict()
assert (saved.text_clean.to_numpy() == cleaned.text_clean.to_numpy()).all(), "script output != notebook output"
assert (saved.text_clean.str.len() > 0).all()
print("script output is identical to the notebook's in-memory result")
display(saved.head(3))
display(pd.read_json(ROOT / "reports" / "cleaning_stats.json", typ="series").to_frame("value"))

Cleaned 54,263 comments in 39s
  saved  -> /home/claude/build/repo/data/processed/goemotions_clean.csv
  stats  -> /home/claude/build/repo/reports/cleaning_stats.json
  rows: 54263
  rows_per_split: {'train': 43410, 'test': 5427, 'validation': 5426}
  vocab_raw_cased: 65523
  vocab_raw_lowercased: 56964
  vocab_clean: 22405
  vocab_reduction_pct_vs_raw_cased: 65.8
  tokens_raw: 695858
  tokens_clean: 475269
  mean_words_raw: 12.82
  mean_words_clean: 8.76
  empty_after_cleaning: 0

(54263, 10) ['comment_id', 'split', 'text', 'text_norm', 'text_clean', 'label_ids', 'label_names', 'caps_words', 'exclaims', 'questions']


script output is identical to the notebook's in-memory result


,comment_id,split,text,text_norm,text_clean,label_ids,label_names,caps_words,exclaims,questions
0,eebbqej,train,My favourite food is anything I didn't have to cook myself.,my favourite food is anything i didn't have to cook myself.,my favourite food anything i not cook myself,27,neutral,0,0,0
1,ed00q6i,train,"Now if he does off himself, everyone will think hes having a laugh screwing with people instead of actuall...","now if he does off himself, everyone will think hes having a laugh screwing with people instead of actuall...",everyone think laugh screw people instead actually dead,27,neutral,0,0,0
2,eezlygj,train,WHY THE FUCK IS BAYLESS ISOING,why the fuck is bayless isoing,why fuck bayless isoing,2,anger,6,0,0


,value
rows,54263
rows_per_split,"{'train': 43410, 'test': 5427, 'validation': 5426}"
vocab_raw_cased,65523
vocab_raw_lowercased,56964
vocab_clean,22405
vocab_reduction_pct_vs_raw_cased,65.8
tokens_raw,695858
tokens_clean,475269
mean_words_raw,12.82
mean_words_clean,8.76


**What just happened.** The cell above ran `python -m src.run_cleaning` — exactly the command a teammate would run after a fresh clone — as a separate process, then loaded its output file and **asserted that it is identical to the cleaning done in this notebook**, that the split sizes are preserved and that no row is empty. So the notebook and the script cannot silently drift apart.

Two practical details worth remembering:

* the CSV must be read with `keep_default_na=False`; otherwise pandas turns a comment that is literally `NA`, `null` or `None` into a missing value;
* `data/processed/` is git-ignored — it is regenerated by the script, so only code, not data, goes into version control.

In [9]:
names = emotion_names()
train, val, test = (saved[saved.split == s] for s in ("train", "validation", "test"))
Y_train = multi_hot(train, names)
print(train.shape, val.shape, test.shape, "| target matrix:", Y_train.shape)
print("TF-IDF models read  text_clean:", train.text_clean.iloc[2])
print("BERT reads          text_norm :", train.text_norm.iloc[2])

(43410, 10) (5426, 10) (5427, 10) | target matrix: (43410, 28)
TF-IDF models read  text_clean: why fuck bayless isoing
BERT reads          text_norm : why the fuck is bayless isoing


## How Phase 2 will use the output

| Model family | Input column | Why |
|---|---|---|
| TF-IDF + logistic regression / SVM / Naive Bayes | `text_clean` | small, de-noised vocabulary |
| BERT (`bert-base-uncased`, lower-cased input matches the model) | `text_norm` | natural text, but with placeholders, quotes and emoji normalised |
| optional numeric features | `caps_words`, `exclaims`, `questions` | style signals removed by case folding |
| targets | `label_ids` → `multi_hot()` | 28 binary columns, multi-label |

**Known caveats to carry into Phase 2:** (1) the class imbalance from 2.1 (part 1) — use macro-F1 and consider class weights; (2) the 71 duplicate texts shared across splits from 2.2; (3) the clean text is lemmatized by a POS tagger that is imperfect on noisy text.

*Reproduce everything:* `pip install -r requirements.txt`, `python -m spacy download en_core_web_sm`, `python -m src.run_cleaning` (see `README.md`).